# LM-weighted-vote dialect ID -- an eleventh method

An alternative to `11_hmm_sequential.ipynb`'s fixed "sticky" transition
matrix, discussed in-session: instead of hand-setting `P(stay in same
class)`, use a **real language model's predicted next-token
distribution** as a context-aware, learned replacement for that fixed
assumption -- `LM_DiD/`, this project's own decoder-only word-level
Transformer (RoPE, RMSNorm, SwiGLU, ... see `LM_DiD/scripts/model.py`).

**"Assume we have a model"**: this notebook uses whatever checkpoint
`LM_DiD/models/` currently has (auto-selects the highest step), and is
built to work correctly regardless of how well-trained that checkpoint
is -- the point here is validating the *pipeline*, not claiming a
mature result. As of writing, the available checkpoint is only a few
thousand steps in (see `LM_DiD/guide.md`'s "Verified working" section)
-- expect this method's accuracy to reflect that (closer to a proof of
concept than a competitive number yet), and to improve as `LM_DiD` gets
trained further, with no changes needed here.

**The mechanism** (per the discussion, formalized):
1. **Word→class table** `P(class | word)`: same idea as the HMM's/
   word_cluster's emission tally -- count each word's occurrences per
   document-level label across `train.jsonl`, Laplace-smoothed,
   normalized to a distribution. Built once per script group, indexed by
   **`LM_DiD`'s own word-level vocabulary ids** (not a separate ad-hoc
   word list) so it can be matrix-multiplied against the LM's output
   directly.
2. **Score a document**: feed its actual words through the LM
   (teacher-forced, causal), take the softmax over the **full vocabulary**
   at every position (`LM_DiD` uses full softmax, not sampled/
   hierarchical), and project each position's predicted distribution
   through the word→class table:
   `expected_class_dist_t = softmax(logits_t) @ word_to_class_table`
   -- one matrix-vector product per position (cheap: `(1, padded_vocab)
   @ (padded_vocab, num_classes)`).
3. **Aggregate**: sum `expected_class_dist_t` over every position in the
   document, argmax -> predicted class. This uses the LM's *expectation*
   of what continues the text, not just the class of whichever word
   actually shows up next -- the point being that even an individually
   uninformative or unseen actual word doesn't lose the signal, since
   the LM's belief about the position was shaped by everything before it.

Same script-gating (`code_switch`/`other` via the deterministic regex)
and `train.jsonl`/`test.jsonl` split as every other method here. Run
this notebook's kernel from the GPU venv (`ai-gpu`).


In [1]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path.cwd().resolve().parents[1]  # project root, when running from Dialect_Identification/notebooks/
DATA_DIR = ROOT / "Dialect_Identification" / "data"
LM_DIR = ROOT / "LM_DiD"

sys.path.insert(0, str(LM_DIR / "scripts"))
from model import DecoderLM  # noqa: E402

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


## 1. Script-gating + data loading (same convention as every other method here)

In [2]:
_MENTION_WITH_FRAGMENT_RE = re.compile(r"\[MENTION\](\s*-[^\s]{1,10})?")
_URL_RE = re.compile(r"\[URL\]")
_INLINE_WHITESPACE_RE = re.compile(r"[ \t]+")
_ARABIC_RE = re.compile(r"[\u0600-\u06ff\u0750-\u077f\u08a0-\u08ff\ufb50-\ufdff\ufe70-\ufeff]")
_LATIN_RE = re.compile(r"[a-zA-Z\u00c0-\u024f]")

ARABIC_CLASSES = ["msa", "darija"]
LATIN_CLASSES = ["arabize", "french", "english"]
GROUP_CLASSES = {"arabic": ARABIC_CLASSES, "latin": LATIN_CLASSES}

MAX_WORDS = 100  # keeps <bos> + words safely under the LM's max_seq_len (128)


def clean_for_classification(text: str) -> str:
    text = _MENTION_WITH_FRAGMENT_RE.sub("", text)
    text = _URL_RE.sub("", text)
    return _INLINE_WHITESPACE_RE.sub(" ", text).strip()


def script_of(text: str) -> str:
    has_ar = bool(_ARABIC_RE.search(text))
    has_lat = bool(_LATIN_RE.search(text))
    if has_ar and has_lat:
        return "mixed"
    if has_ar:
        return "arabic"
    if has_lat:
        return "latin"
    return "other"


def load_group_rows(split: str, group: str) -> list[dict]:
    rows = []
    with (DATA_DIR / f"{split}.jsonl").open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            cleaned = clean_for_classification(r["text"])
            if script_of(cleaned) != group:
                continue
            rows.append({"id": r["id"], "text": r["text"], "label": r["label"]})
    return rows


## 2. Load `LM_DiD`'s vocabulary + latest checkpoint (frozen)

Auto-selects the highest-step checkpoint present, same convention as
every embedding-checkpoint loader elsewhere in this project.

In [3]:
vocab = json.loads((LM_DIR / "data" / "vocab.json").read_text(encoding="utf-8"))
word_to_id = vocab["word_to_id"]
pad_id, unk_id, bos_id, eos_id = (word_to_id[s] for s in vocab["specials"])
print(f"LM vocab: {vocab['vocab_size']:,} words (incl. specials)")

checkpoints = sorted(
    (LM_DIR / "models").glob("checkpoint_step*.pt"),
    key=lambda p: int(p.stem.split("step")[-1]),
)
CHECKPOINT = checkpoints[-1]
ckpt = torch.load(CHECKPOINT, map_location=device)
ckpt_args = ckpt["args"]
print(f"Using checkpoint: {CHECKPOINT.name} (step={ckpt['step']:,})")

lm = DecoderLM(
    vocab_size=vocab["vocab_size"],
    d_model=ckpt_args["d_model"],
    num_heads=ckpt_args["num_heads"],
    num_layers=ckpt_args["num_layers"],
    max_seq_len=ckpt_args["seq_len"],
    dropout=0.0,  # eval mode -- no dropout regardless of what training used
    pad_id=pad_id,
).to(device)
lm.load_state_dict(ckpt["model_state_dict"])
lm.eval()
for p in lm.parameters():
    p.requires_grad = False

PADDED_VOCAB_SIZE = lm.padded_vocab_size
print(f"Model: padded_vocab_size={PADDED_VOCAB_SIZE:,}, max_seq_len={ckpt_args['seq_len']}")


def words_to_ids(text: str) -> list[int]:
    words = text.split()[:MAX_WORDS]
    return [bos_id] + [word_to_id.get(w, unk_id) for w in words]


LM vocab: 30,004 words (incl. specials)


Using checkpoint: checkpoint_step690000.pt (step=690,000)
Model: padded_vocab_size=30,016, max_seq_len=128


## 3. Word -> class table, per group

Same tally idea as the HMM/word_cluster emissions (Laplace-smoothed,
normalized per word), but indexed by `LM_DiD`'s own vocabulary ids so it
can be matrix-multiplied against the LM's predicted distribution
directly. A `(padded_vocab_size, num_classes)` dense array -- tiny
(30,016 x <=3), no reason to keep it sparse.

In [4]:
ALPHA = 0.5  # Laplace/add-alpha smoothing


def build_word_to_class_table(rows: list[dict], classes: list[str]) -> np.ndarray:
    label_to_id = {c: i for i, c in enumerate(classes)}
    counts = np.zeros((PADDED_VOCAB_SIZE, len(classes)), dtype=np.float64)
    for r in rows:
        class_idx = label_to_id[r["label"]]
        for w in r["text"].split():
            tok_id = word_to_id.get(w, unk_id)
            if tok_id < PADDED_VOCAB_SIZE:
                counts[tok_id, class_idx] += 1

    counts += ALPHA
    table = counts / counts.sum(axis=1, keepdims=True)
    return table.astype(np.float32)


word_to_class_tables = {}
for group in ("arabic", "latin"):
    train_rows = load_group_rows("train", group)
    table = build_word_to_class_table(train_rows, GROUP_CLASSES[group])
    word_to_class_tables[group] = torch.from_numpy(table).to(device)
    print(f"{group}: table built from {len(train_rows):,} train rows, shape={tuple(table.shape)}")


arabic: table built from 10,000 train rows, shape=(30016, 2)


latin: table built from 6,848 train rows, shape=(30016, 3)


## 4. Score a document: LM next-token distribution -> class vote, summed over positions

In [5]:
@torch.no_grad()
def score_document(text: str, group: str) -> np.ndarray:
    ids = words_to_ids(text)
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)  # (1, seq_len)
    logits, _ = lm(input_ids)  # (1, seq_len, padded_vocab_size)
    probs = F.softmax(logits[0], dim=-1)  # (seq_len, padded_vocab_size)

    table = word_to_class_tables[group]  # (padded_vocab_size, num_classes)
    per_position_votes = probs @ table  # (seq_len, num_classes)
    return per_position_votes.sum(dim=0).cpu().numpy()


def classify(text: str, group: str) -> str:
    classes = GROUP_CLASSES[group]
    scores = score_document(text, group)
    return classes[int(scores.argmax())]


## 5. Evaluate on `test.jsonl`

In [6]:
all_results = {}
for group in ("arabic", "latin"):
    classes = GROUP_CLASSES[group]
    test_rows = load_group_rows("test", group)
    y_true = [r["label"] for r in test_rows]
    y_pred = [classify(r["text"], group) for r in test_rows]
    acc = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
    all_results[group] = dict(rows=test_rows, y_true=y_true, y_pred=y_pred, accuracy=acc)

    print(f"\n{group}: accuracy={acc:.4f} ({sum(t == p for t, p in zip(y_true, y_pred))}/{len(y_true)})")
    print("Per-class accuracy:")
    for cname in classes:
        idx = [i for i, t in enumerate(y_true) if t == cname]
        n_correct = sum(y_pred[i] == cname for i in idx)
        print(f"  {cname:<10} {n_correct:>4}/{len(idx):<4} acc={n_correct / len(idx) if idx else float('nan'):.3f}")



arabic: accuracy=0.7315 (1828/2499)
Per-class accuracy:
  msa         719/809  acc=0.889
  darija     1109/1690 acc=0.656



latin: accuracy=0.8148 (1395/1712)
Per-class accuracy:
  arabize     843/926  acc=0.910
  french      517/635  acc=0.814
  english      35/151  acc=0.232


In [7]:
def print_confusion(classes, y_true, y_pred, title):
    n = len(classes)
    idx = {c: i for i, c in enumerate(classes)}
    mat = [[0] * n for _ in range(n)]
    for t, p in zip(y_true, y_pred):
        mat[idx[t]][idx[p]] += 1
    print(f"\n{title} confusion matrix (rows=true, cols=predicted):")
    print("true\\pred" + "".join(f"{c:>12}" for c in classes))
    for i, c in enumerate(classes):
        print(f"{c:<10}" + "".join(f"{mat[i][j]:>12}" for j in range(n)))


for group, res in all_results.items():
    print_confusion(GROUP_CLASSES[group], res["y_true"], res["y_pred"], group)



arabic confusion matrix (rows=true, cols=predicted):
true\pred         msa      darija
msa                719          90
darija             581        1109

latin confusion matrix (rows=true, cols=predicted):
true\pred     arabize      french     english
arabize            843          82           1
french             116         517           2
english            113           3          35


## 6. Misclassified examples

In [8]:
import random

random.seed(0)

for group, res in all_results.items():
    wrong = [
        (r["text"], t, p) for r, t, p in zip(res["rows"], res["y_true"], res["y_pred"]) if t != p
    ]
    print(f"\n=== {group}: {len(wrong)}/{len(res['rows'])} misclassified ({len(wrong) / len(res['rows']):.1%}) ===")
    for text, true_label, pred_label in random.sample(wrong, min(10, len(wrong))):
        snippet = text[:140] + ("..." if len(text) > 140 else "")
        print(f"\n  true={true_label}  pred={pred_label}")
        print(f"  {snippet!r}")



=== arabic: 671/2499 misclassified (26.9%) ===

  true=msa  pred=darija
  'البيرة والويسكي تحيا الجزائر'

  true=darija  pred=msa
  ': مكانش ليحبلك الخير اكثر من الأم ربي يحفضلنا ماتنا وماتكم🙏 والله يرحم المتوفين ❤'

  true=msa  pred=darija
  'أهلا بالربيع'

  true=darija  pred=msa
  'السلام عليكم\nمعذرة ايتها الاخت الكريمة\nواش نقول هذا اسلوبي وهذه طريقتي\nفاسف اخوك معتذرة\nيا حاملة كتاب الله\nالله يوفقك لختم حفظ كتابه\nثم الله...'

  true=darija  pred=msa
  'شفت ربی کپفاش پعاقب المفسدپن فی الارض الی فپه پکفپه'

  true=darija  pred=msa
  'مشاء الله جميل'

  true=darija  pred=msa
  'ربي يبين الحق أو ياخذو المجرمين عقابهم أو ربي ينصر هاد العائلة مسكينة إبانو ناس ملاح ربي إصبركم'

  true=darija  pred=msa
  '[MENTION] ماش إنتقاد هدا ،يسمى اعتراض على حكم ربي و حقد و عدم القناعة بما آتاهم الله و سوء الظن ،أقرا تعليق لي ريبوندالي خونا ڨالك أين عدالة...'

  true=darija  pred=msa
  'شكرآ جزيلا على الموضوع الرائع و المميز واصل تالقك معنا في المنتدى بارك الله فيك اخي ... ننتظر منك الكثير من خلا

## 7. Compared against every other method tried on this task

Includes the HMM (`11_hmm_sequential.ipynb`) this method was proposed as
an alternative to -- both replace the same "how much does the previous
context constrain the next word's class" piece of the pipeline, one with
a fixed sticky transition matrix, the other with a learned LM.

In [9]:
comparison = {
    "regex + real fastText lid.176": (0.340, 0.630),
    "word-cluster baseline (cluster-scored)": (0.562, 0.562),
    "fastText-style (cluster-scored)": (0.582, 0.582),
    "regex/marker-word baseline": (0.699, 0.748),
    "cluster + 200 human-labeled prototypes": (0.701, 0.810),
    "Cavnar & Trenkle (n-gram profiles)": (0.754, 0.844),
    "KNN on sentence embeddings (best of 32 configs)": (0.770, 0.835),
    "cluster + majority vote (best k)": (0.796, 0.846),
    "classifier head (transformer)": (0.819, 0.863),
    "n-gram + SVM-RBF (best of notebook 04)": (0.833, 0.881),
    "LM-weighted-vote (this notebook)": (all_results["arabic"]["accuracy"], all_results["latin"]["accuracy"]),
}

print(f"{'method':<50}{'arabic':>10}{'latin':>10}")
for name, (a, l) in sorted(comparison.items(), key=lambda kv: kv[1][0]):
    print(f"{name:<50}{a:>10.3f}{l:>10.3f}")


method                                                arabic     latin
regex + real fastText lid.176                          0.340     0.630
word-cluster baseline (cluster-scored)                 0.562     0.562
fastText-style (cluster-scored)                        0.582     0.582
regex/marker-word baseline                             0.699     0.748
cluster + 200 human-labeled prototypes                 0.701     0.810
LM-weighted-vote (this notebook)                       0.731     0.815
Cavnar & Trenkle (n-gram profiles)                     0.754     0.844
KNN on sentence embeddings (best of 32 configs)        0.770     0.835
cluster + majority vote (best k)                       0.796     0.846
classifier head (transformer)                          0.819     0.863
n-gram + SVM-RBF (best of notebook 04)                 0.833     0.881
